# Notebook 1 — What Makes a Good Feature?

**Difficulty:** ⭐⭐ | **Estimated time:** 35 min

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Define **three criteria** for a good feature: predictive power, independence, and interpretability.
2. Distinguish **signal features** from **noise features** experimentally.
3. Explain the **"garbage in, garbage out"** principle and its practical consequences.

---

**Week 12 Theme — Airbnb Listing Price Prediction**

Throughout Week 12 we work with a synthetic but realistic Airbnb dataset. Every listing has numeric, categorical, datetime, and text attributes. Our job is to decide *which* of those attributes are worth handing to a model — and how to engineer better ones from what we have.

## The Detective Analogy

Think of your model as a **detective** trying to solve the mystery: *"How much should this listing cost per night?"*

You hand the detective a set of **clues** — these are your features. A good clue has three properties:

| Property | What it means | Good example | Bad example |
|---|---|---|---|
| **Relevant** | Predicts the outcome | Number of bedrooms | Customer's favourite colour |
| **Non-redundant** | Adds new information beyond other clues | Distance to city centre | `accommodates` when you already have `bedrooms` |
| **Interpretable** | The detective understands what it means | `review_score` | PCA component 7 |

A detective handed 200 irrelevant clues alongside 5 good ones will still solve the case — but it takes much longer and they might chase the wrong lead. This is exactly what happens to your model.

> **"Garbage in, garbage out"** — No matter how clever the algorithm, it cannot extract a signal that isn't in the features you provide.

## Why Does This Matter for ML?

- **Features dominate model performance** more than algorithm choice. Switching from a linear model to a random forest on bad features gives modest gains. Switching from bad features to good features on a linear model can double your R².
- **80 % of Kaggle winning solutions** win through feature engineering, not model tuning. The top-ranked notebooks spend most of their code in the feature section, not the model section.
- **Debugging** becomes impossible without interpretable features. If your model makes a bad prediction and your features are PCA components, you have no idea where the error came from.
- **Regulatory compliance** in finance, healthcare, and hiring law requires you to explain your model's inputs. "The model used PCA component 3" is not an acceptable explanation.

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(42)          # reproducibility for every cell that follows
sns.set_theme(style='whitegrid')  # consistent plot style throughout

print('Libraries loaded successfully.')

## Criterion 1 — Predictive Power

A feature has **predictive power** if knowing its value helps you guess the target, either on its own or in combination with other features.

The simplest measure: **Pearson correlation** with the target for numeric features. But correlation only catches *linear* relationships. A feature can be highly predictive yet have near-zero correlation (e.g. a U-shaped relationship).

We will look at correlation first, then go beyond it in later notebooks.

**The Airbnb dataset** we build below has:
- `bedrooms`, `bathrooms`, `accommodates` — capacity features (correlated with each other)
- `review_score` — quality signal
- `distance_to_centre` — location (negative effect on price)
- `host_experience_days` — proxy for host quality
- `is_superhost` — binary quality flag
- Several pure noise columns
- Target: `price` (per night, in USD)

In [ ]:
np.random.seed(42)

N = 2000  # number of synthetic listings

# ── Capacity features (intentionally correlated with each other) ───────────
bedrooms          = np.random.randint(1, 7, N).astype(float)
accommodates      = bedrooms * 2 + np.random.randint(0, 3, N)  # accommodates ~ 2*bedrooms
bathrooms         = np.round(bedrooms * 0.6 + np.random.uniform(0, 0.5, N), 1)

# ── Quality / reputation features ─────────────────────────────────────────
review_score      = np.clip(np.random.normal(4.5, 0.4, N), 1, 5)
host_experience_days = np.random.exponential(500, N).astype(int)  # skewed right
is_superhost      = (review_score > 4.7).astype(int)              # superhosts have high scores

# ── Location ───────────────────────────────────────────────────────────────
distance_to_centre = np.abs(np.random.normal(3.5, 2.0, N))  # km from city centre

# ── Pure noise features ────────────────────────────────────────────────────
noise_1 = np.random.randn(N)           # completely random
noise_2 = np.random.randint(0, 100, N) # random integers
noise_3 = np.random.choice(['A','B','C'], N)  # random categories (we'll encode later)

# ── Target: price (USD per night) ─────────────────────────────────────────
# True data-generating process (the model will never see this formula):
price = (
    40
    + 25  * bedrooms
    + 15  * bathrooms
    + 18  * review_score
    - 5   * distance_to_centre
    + 0.02 * host_experience_days
    + 20  * is_superhost
    + np.random.normal(0, 15, N)         # irreducible noise
)
price = np.clip(price, 20, 600)          # realistic price bounds

# ── Bundle into a DataFrame ────────────────────────────────────────────────
df = pd.DataFrame({
    'bedrooms':             bedrooms,
    'bathrooms':            bathrooms,
    'accommodates':         accommodates,
    'review_score':         review_score,
    'distance_to_centre':   distance_to_centre,
    'host_experience_days': host_experience_days,
    'is_superhost':         is_superhost,
    'noise_1':              noise_1,
    'noise_2':              noise_2,
    'price':                price
})

print(f'Dataset shape: {df.shape}')
print('\nFirst five rows:')
df.head()

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Correlation matrix ────────────────────────────────────────────────────
numeric_cols = [c for c in df.columns]  # all columns are numeric here
corr = df.corr()                         # Pearson correlations

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle (mirror)
sns.heatmap(
    corr,
    mask=mask,
    annot=True,        # show correlation numbers inside cells
    fmt='.2f',
    cmap='RdYlGn',     # red = negative, green = positive
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Pearson Correlation Matrix — Airbnb Features', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# ── Highlight correlations with price ─────────────────────────────────────
price_corr = corr['price'].drop('price').sort_values(ascending=False)
print('\nCorrelation with price (sorted):')
print(price_corr.to_string())
print('\nObservation: bedrooms, bathrooms, accommodates, review_score, and is_superhost')
print('have the highest |correlation| with price. noise_1 and noise_2 are near zero.')

## Criterion 2 — Independence / Low Redundancy

If two features are highly correlated with each other, they carry largely the **same information**. Adding both to a model:

- Does **not** improve tree models much — trees already split on whichever is most useful.
- Can cause **multicollinearity** in linear models — the coefficients become unstable and hard to interpret. Two correlated features "fight" over the coefficient.

**Variance Inflation Factor (VIF)** measures how much the variance of a coefficient is inflated by collinearity:

$$\text{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is the R² of regressing feature $j$ on all other features.

- VIF = 1 → no collinearity (feature is orthogonal to others)
- VIF = 5 → moderate (some collinearity, watch it)
- VIF > 10 → severe (feature is highly predictable from others — consider dropping)

Notice that `bedrooms`, `accommodates`, and `bathrooms` were constructed to be correlated with each other — we expect high VIF for these.

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Pairplot of capacity features (shows their correlation visually) ──────
capacity_cols = ['bedrooms', 'bathrooms', 'accommodates', 'price']
fig, axes = plt.subplots(len(capacity_cols), len(capacity_cols), figsize=(10, 10))

# Use seaborn pairplot (creates its own figure, then we annotate)
pairplot_fig = sns.pairplot(
    df[capacity_cols],
    diag_kind='kde',        # diagonal: kernel density estimate
    plot_kws={'alpha': 0.3, 's': 15},  # semi-transparent points
    corner=True             # only lower triangle
)
pairplot_fig.figure.suptitle(
    'Capacity features are highly correlated with each other\n'
    '(bedrooms ↔ accommodates ↔ bathrooms)',
    y=1.02, fontsize=13
)
plt.show()

# ── Compute VIF for each feature ──────────────────────────────────────────
feature_cols = [
    'bedrooms', 'bathrooms', 'accommodates',
    'review_score', 'distance_to_centre',
    'host_experience_days', 'is_superhost',
    'noise_1', 'noise_2'
]

X_vif = df[feature_cols].values.astype(float)

vif_data = pd.DataFrame()
vif_data['feature'] = feature_cols
vif_data['VIF'] = [
    variance_inflation_factor(X_vif, i)   # statsmodels function
    for i in range(X_vif.shape[1])
]
vif_data = vif_data.sort_values('VIF', ascending=False)

# ── Bar chart of VIF values ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if v > 10 else '#ff7f0e' if v > 5 else '#2ca02c'
          for v in vif_data['VIF']]                        # red=severe, orange=moderate, green=ok
ax.barh(vif_data['feature'], vif_data['VIF'], color=colors)
ax.axvline(10, color='red',    linestyle='--', linewidth=1.5, label='VIF=10 (severe threshold)')
ax.axvline(5,  color='orange', linestyle='--', linewidth=1.5, label='VIF=5  (moderate threshold)')
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factor by Feature')
ax.legend()
plt.tight_layout()
plt.show()

print('\nVIF Table:')
print(vif_data.to_string(index=False))
print('\nFeatures with VIF > 10 are potentially redundant (high collinearity):')
print(vif_data[vif_data['VIF'] > 10]['feature'].tolist())

## Criterion 3 — Interpretability

Can a **human** understand what the feature means?

Interpretable features are valuable for:
- **Debugging**: if the model makes a strange prediction, you can trace it back to a meaningful input.
- **Stakeholder trust**: a non-technical manager can sanity-check "the model says distance to city centre matters" but not "the model uses PCA component 4".
- **Regulatory compliance**: GDPR, fair lending laws, and medical device regulations often require explainability.

**PCA** (Principal Component Analysis) is a powerful dimensionality reduction technique, but PCA components are linear combinations of all original features — they are mathematically optimal but humanly opaque.

Below we compare two feature sets on the same Ridge regression:
1. **Interpretable**: `bedrooms`, `review_score`, `distance_to_centre`
2. **PCA**: top 3 principal components of the full feature matrix

We'll see similar R² — but only the interpretable set tells a story.

In [ ]:
np.random.seed(42)

X_full = df[feature_cols].values.astype(float)
y      = df['price'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)

# ── Feature set 1: interpretable (3 hand-picked features) ─────────────────
interp_idx = [feature_cols.index(c) for c in ['bedrooms', 'review_score', 'distance_to_centre']]
X_train_interp = X_train[:, interp_idx]
X_test_interp  = X_test[:, interp_idx]

ridge_interp = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
ridge_interp.fit(X_train_interp, y_train)
r2_interp = ridge_interp.score(X_test_interp, y_test)

# Extract interpretable coefficients for display
coefs_interp = ridge_interp.named_steps['ridge'].coef_

# ── Feature set 2: top-3 PCA components ───────────────────────────────────
scaler_pca = StandardScaler()
X_train_scaled = scaler_pca.fit_transform(X_train)
X_test_scaled  = scaler_pca.transform(X_test)

pca = PCA(n_components=3, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)   # 3 abstract components
X_test_pca  = pca.transform(X_test_scaled)

ridge_pca = Ridge(alpha=1.0)
ridge_pca.fit(X_train_pca, y_train)
r2_pca = ridge_pca.score(X_test_pca, y_test)

# Extract PCA coefficients (meaningless to a human)
coefs_pca = ridge_pca.coef_

# ── Print comparison ───────────────────────────────────────────────────────
print('='*55)
print('Ridge Regression — Feature Set Comparison')
print('='*55)
print(f'\nInterpretable features  R² on test set: {r2_interp:.3f}')
print('  Coefficients (after scaling):')
for name, coef in zip(['bedrooms', 'review_score', 'distance_to_centre'], coefs_interp):
    print(f'    {name:25s}: {coef:+.2f}')   # sign and magnitude tell a clear story

print(f'\nPCA components (top 3)  R² on test set: {r2_pca:.3f}')
print('  Coefficients (after PCA):')
for i, coef in enumerate(coefs_pca):
    print(f'    PC{i+1:1d}                      : {coef:+.2f}')  # numbers mean nothing to a human

print('\n→ Similar R², but interpretable features tell a clear causal story:')
print('  More bedrooms → higher price (+); better reviews → higher price (+);')
print('  farther from centre → lower price (−).')
print('  PCA coefficients are mathematically valid but humanly opaque.')

## Signal vs. Noise — The Curse of Dimensionality

What happens when we **pad** our feature set with random noise columns?

- **Unregularised linear regression** — each noise feature gets a small but non-zero coefficient. The model overfits, and out-of-sample performance degrades as noise grows.
- **Ridge regression** — shrinks all coefficients towards zero. Noise features get near-zero weights and do less damage.
- **Random forest** — feature importance is diluted but trees can still find signal. Degradation is milder, but computation cost grows.

The **curse of dimensionality**: as the number of features $p$ grows relative to the number of samples $n$, every pair of points becomes roughly equidistant. Distance-based methods (KNN, SVMs with RBF) suffer first; linear models with $p > n$ become under-determined.

Rule of thumb: you need at least **10–50 samples per feature** to avoid serious overfitting in a linear model.

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Base feature matrix (the 7 real features) ─────────────────────────────
real_feature_cols = [
    'bedrooms', 'bathrooms', 'accommodates',
    'review_score', 'distance_to_centre',
    'host_experience_days', 'is_superhost'
]
X_base = df[real_feature_cols].values.astype(float)
y = df['price'].values

# ── Models to compare ─────────────────────────────────────────────────────
models = {
    'LinearRegression': LinearRegression(),
    'Ridge (alpha=10)':  Ridge(alpha=10),
    'RandomForest':      RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

noise_counts = [0, 5, 20, 50, 100]  # how many random noise columns to add
results = {name: [] for name in models}

for n_noise in noise_counts:
    # Add n_noise columns of pure random Gaussian noise
    if n_noise == 0:
        X_aug = X_base
    else:
        noise_matrix = np.random.randn(len(y), n_noise)  # pure noise
        X_aug = np.hstack([X_base, noise_matrix])        # append to real features

    for name, model in models.items():
        pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
        # 5-fold CV — negative MSE → R² (cv scoring)
        cv_scores = cross_val_score(pipe, X_aug, y, cv=5, scoring='r2')
        results[name].append(cv_scores.mean())           # store mean CV R²

# ── Plot R² vs. number of noise features ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

markers = ['o-', 's--', '^:']  # different line styles per model
for (name, r2_list), marker in zip(results.items(), markers):
    ax.plot(noise_counts, r2_list, marker, linewidth=2, markersize=7, label=name)

ax.set_xlabel('Number of Noise Features Added', fontsize=12)
ax.set_ylabel('Cross-validated R²', fontsize=12)
ax.set_title('Effect of Adding Pure Noise Features on Validation R²', fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(noise_counts)
plt.tight_layout()
plt.show()

print('\nObservations:')
print('  LinearRegression degrades the most — it blindly uses every feature.')
print('  Ridge is more robust — it penalises large weights on noise features.')
print('  RandomForest degrades least — feature importance naturally ignores noise.')

## Feature Relevance Categories

Not all features are equal. Here is a practical taxonomy:

| Category | Description | Example |
|---|---|---|
| **Directly predictive** | High individual correlation with target | `bedrooms` |
| **Indirectly predictive** | Low alone, but useful in combination | `is_superhost` alone is weak; `is_superhost × review_score` is strong |
| **Conditionally predictive** | Predictive only for a subset of data | `pool_available` only matters in summer listings |
| **Noise** | No relationship with target | `noise_1`, `noise_2` |

The interaction category is the most commonly missed. Many features that look unimportant in a univariate analysis become powerful when combined with another feature.

**Airbnb example:** `is_superhost` alone adds only a modest price premium. But *superhost with very high review score* commands a significant premium — the two features interact.

In [ ]:
np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Create the interaction feature ────────────────────────────────────────
df['superhost_x_review'] = df['is_superhost'] * df['review_score']

# ── Visualise: scatter of review_score vs price, coloured by superhost ────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: raw relationship
for sh_val, label, color in [(0, 'Not superhost', '#5b9bd5'), (1, 'Superhost', '#ed7d31')]:
    mask = df['is_superhost'] == sh_val
    axes[0].scatter(
        df.loc[mask, 'review_score'],
        df.loc[mask, 'price'],
        alpha=0.3, s=15, label=label, color=color
    )
axes[0].set_xlabel('Review Score')
axes[0].set_ylabel('Price (USD/night)')
axes[0].set_title('Review Score vs Price\n(coloured by superhost status)')
axes[0].legend()

# Right: interaction feature vs price
axes[1].scatter(
    df['superhost_x_review'], df['price'],
    alpha=0.3, s=15, color='#70ad47'
)
axes[1].set_xlabel('Interaction: is_superhost × review_score')
axes[1].set_ylabel('Price (USD/night)')
axes[1].set_title('Interaction Feature vs Price\n(captures synergy explicitly)')

plt.suptitle('Conditional Feature: Superhost × Review Score', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── Compare R² with and without the interaction feature ───────────────────
base_feats = ['is_superhost', 'review_score', 'bedrooms', 'distance_to_centre']
X_no_int = df[base_feats].values.astype(float)
X_with_int = np.column_stack([X_no_int, df['superhost_x_review'].values])
y = df['price'].values

for label, X in [('Without interaction', X_no_int), ('With interaction feature', X_with_int)]:
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
    cv_r2 = cross_val_score(pipe, X, y, cv=5, scoring='r2').mean()
    print(f'{label:30s}: CV R² = {cv_r2:.4f}')

## The Feature Engineering Mindset

The best features come from **domain knowledge**, not from statistical formulas.

Ask yourself: *"If I were estimating this listing's price manually, what would I look at?"*

A human Airbnb host thinks:
- *"My listing has 4 bedrooms but only accommodates 5 — that's inefficient space usage."*
- *"I have 500 reviews and they're all 5 stars — that's more credible than 2 reviews and a 5-star rating."*
- *"I'm 1 km from the Eiffel Tower — that's a premium location."*

These intuitions translate directly into **engineered features** that often outperform raw inputs.

In [ ]:
np.random.seed(42)

# ── Engineer domain-expert features ───────────────────────────────────────

# 1. Capacity efficiency: how many people per bedroom?
#    A listing with 3 bedrooms but 8 guests is "efficient" and may charge more per person
df['guests_per_bedroom'] = df['accommodates'] / df['bedrooms'].clip(lower=1)

# 2. Review credibility: high score on many reviews is more trustworthy than few reviews
#    We don't have num_reviews in our dataset, so we approximate with a synthetic column
np.random.seed(42)
df['num_reviews'] = np.random.negative_binomial(n=5, p=0.05, size=N)  # typical review counts
df['review_credibility'] = df['review_score'] * np.log1p(df['num_reviews'])  # log dampens outliers

# 3. Host maturity score: experienced hosts with great scores command premiums
df['host_maturity'] = np.log1p(df['host_experience_days']) * df['review_score']

# ── Baseline: raw features only ───────────────────────────────────────────
X_raw = df[[
    'bedrooms', 'bathrooms', 'accommodates',
    'review_score', 'distance_to_centre',
    'host_experience_days', 'is_superhost'
]].values.astype(float)

# ── Enhanced: raw + domain-expert features ────────────────────────────────
X_enhanced = df[[
    'bedrooms', 'bathrooms', 'accommodates',
    'review_score', 'distance_to_centre',
    'host_experience_days', 'is_superhost',
    'guests_per_bedroom', 'review_credibility', 'host_maturity'
]].values.astype(float)

y = df['price'].values

for label, X in [('Raw features only   ', X_raw), ('Raw + domain-expert features', X_enhanced)]:
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=1.0))])
    cv_r2 = cross_val_score(pipe, X, y, cv=5, scoring='r2').mean()
    print(f'{label}: CV R² = {cv_r2:.4f}')

print('\nEngineered features provide a small but consistent improvement.')
print('In real datasets with messier relationships, the gain is typically much larger.')

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1.** You have a dataset with 500 samples and 200 features, all of which are random noise except for 5 real predictors. Which model would degrade the *least* as you increase the number of noise features from 0 to 200 — LinearRegression, Ridge, or RandomForest? Why?

<details><summary>Show answer</summary>

**RandomForest** degrades the least because it uses feature importance internally (splitting only on features that reduce impurity) and only considers a random subset of features at each split. Ridge comes second because L2 regularisation shrinks coefficients towards zero — noise features get near-zero weights. UnregularisedLinearRegression degrades the most because it assigns non-zero (overfitted) coefficients to every noise feature, inflating variance on the test set.

</details>

---

**Question 2.** VIF for `accommodates` is 18. Should you definitely drop it? What are two alternatives to dropping?

<details><summary>Show answer</summary>

Not necessarily. VIF > 10 signals *potential* multicollinearity, not a mandatory drop. Two alternatives:
1. **Use Ridge / Lasso regression** — regularisation handles multicollinearity by shrinking correlated coefficients together. You keep all features but avoid unstable coefficients.
2. **Create a composite feature** — e.g. `guests_per_bedroom = accommodates / bedrooms` captures the same information as both columns in a single, more interpretable feature, reducing collinearity.

</details>

---

**Question 3.** A feature has near-zero Pearson correlation with the target. Does this mean it is useless? Give a concrete example of a feature that is predictive yet has low linear correlation.

<details><summary>Show answer</summary>

No. Pearson correlation only measures *linear* association. A feature with a U-shaped, step-function, or interaction relationship with the target can have zero linear correlation yet be highly informative. Example: `distance_to_city_centre` might have low linear correlation with price in a city where both the historic centre AND the trendy outer suburb command high prices — the relationship is U-shaped. A tree model would discover this; Pearson correlation would miss it. The fix is to check mutual information or a scatter plot, not just the correlation coefficient.

</details>

## Key Takeaways

1. **Good features are predictive, non-redundant, and interpretable.** You rarely get all three perfectly — it is always a trade-off.

2. **Engineering beats algorithm-switching.** Before tuning hyperparameters, invest time in understanding your data and creating meaningful features.

3. **VIF > 10** flags collinear features. For tree models, redundancy is less dangerous. For linear models, use regularisation or dimensionality reduction.

4. **Noise features hurt most in unregularised models.** Always use at least Ridge regularisation when you have many features.

5. **Domain knowledge is your most powerful feature engineering tool.** Ask "what would a human expert look at?" before opening a statistical test.

---

**Next up — Notebook 2: Interaction Features.** We will go beyond individual features and learn to capture *synergies* between features explicitly.